In [6]:
from ollama import chat
import json
import time
from typing import Any, List, Optional
from pydantic import BaseModel, Field, ValidationError, ConfigDict, field_validator

image_path = "../data/image-13.png"
VISION_OPTIONS = {"num_gpu": 0}

SCHEMA_COLUMNS = ["Prod date", "Exp date", "Product Name", "Batch No.", "pH"]

class ProductRow(BaseModel):
    model_config = ConfigDict(populate_by_name=True, extra="ignore")

    prod_date: Optional[str] = Field(default=None, alias="Prod date")
    exp_date: Optional[str] = Field(default=None, alias="Exp date")
    product_name: Optional[str] = Field(default=None, alias="Product Name")
    batch_no: Optional[str] = Field(default=None, alias="Batch No.")
    ph: Optional[str] = Field(default=None, alias="pH")

    @field_validator("prod_date", "exp_date", "product_name", "batch_no", "ph", mode="before")
    @classmethod
    def stringify_scalars(cls, value: Any) -> Optional[str]:
        if value is None:
            return None
        if isinstance(value, str):
            return value.strip() or None
        return str(value)

class ProductTable(BaseModel):
    model_config = ConfigDict(extra="ignore")
    rows: List[ProductRow]

def is_transient_ollama_error(error: Exception) -> bool:
    message = str(error).lower()
    transient_markers = [
        "status code: 500",
        "forcibly closed by the remote host",
        "connection reset",
        "wsarecv",
        "unexpected eof",
    ]
    return any(marker in message for marker in transient_markers)

def safe_chat(*, model: str, messages: list, retries: int = 3, base_delay: float = 1.5, **kwargs):
    """Retry transient Ollama transport/runtime failures with exponential backoff."""
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            return chat(model=model, messages=messages, **kwargs)
        except Exception as error:
            last_error = error
            if (attempt == retries) or (not is_transient_ollama_error(error)):
                raise

            delay = base_delay ** attempt
            print(f"[retry {attempt}/{retries}] {error}")
            print(f"Waiting {delay:.1f}s before retrying {model}...")
            time.sleep(delay)

    raise last_error

def extract_json_candidate(text: str) -> str:
    """Extract a JSON-looking substring from model output."""
    stripped = text.strip()
    if stripped.startswith("{"):
        return stripped

    start = stripped.find("{")
    if start == -1:
        return stripped

    return stripped[start:]

def repair_json_text(text: str) -> str:
    """Try simple JSON repair for common LLM truncation issues."""
    candidate = extract_json_candidate(text).strip()
    if not candidate:
        return candidate

    open_curly = candidate.count("{")
    close_curly = candidate.count("}")
    if close_curly < open_curly:
        candidate = candidate + ("}" * (open_curly - close_curly))

    open_square = candidate.count("[")
    close_square = candidate.count("]")
    if close_square < open_square:
        candidate = candidate + ("]" * (open_square - close_square))

    return candidate

def coerce_table_with_pydantic(structured_text: str) -> ProductTable:
    """Validate and coerce LLM output into strict schema using Pydantic."""
    attempts = [structured_text, repair_json_text(structured_text)]

    for attempt in attempts:
        try:
            parsed_json = json.loads(extract_json_candidate(attempt))
            return ProductTable.model_validate(parsed_json)
        except (json.JSONDecodeError, ValidationError):
            continue

    fallback_response = safe_chat(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Return ONLY valid JSON with exact schema: "
                    "{\"rows\": [{\"Prod date\": null, \"Exp date\": null, \"Product Name\": null, \"Batch No.\": null, \"pH\": null}]}. "
                    "No markdown, no explanation, no extra keys."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Fix this malformed JSON and return valid JSON only:\n\n"
                    f"{structured_text}"
                ),
            },
        ],
    )

    fixed_text = fallback_response.message.content
    parsed_fixed = json.loads(repair_json_text(fixed_text))
    return ProductTable.model_validate(parsed_fixed)

def row_to_schema_dict(row: ProductRow) -> dict:
    dumped = row.model_dump(by_alias=True)
    return {
        "Prod date": dumped.get("Prod date"),
        "Exp date": dumped.get("Exp date"),
        "Product Name": dumped.get("Product Name"),
        "Batch No.": dumped.get("Batch No."),
        "pH": dumped.get("pH"),
    }

# Step 1: OCR extraction from the image
ocr_response = safe_chat(
    model="gemma3:4b",
    messages=[
        {
            "role": "user",
            "content": "Extract all table text from this image exactly as it appears.",
            "images": [image_path],
        }
    ],
    options=VISION_OPTIONS,
)

raw_text = ocr_response.message.content
print("=== OCR Raw Output ===")
print(raw_text)

# Step 2: Convert OCR text into strict schema rows
llm_response = safe_chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "system",
            "content": (
                "You extract structured table rows from OCR text. "
                "Return ONLY valid JSON (no markdown, no explanation). "
                "Use this exact schema with exact keys and same case: "
                "{\"rows\": [{\"Prod date\": null, \"Exp date\": null, \"Product Name\": null, \"Batch No.\": null, \"pH\": null}]}. "
                "If multiple table rows exist, include all rows. "
                "If a field is missing or unclear, set it to null."
            ),
        },
        {
            "role": "user",
            "content": f"OCR text:\n\n{raw_text}",
        },
    ],
)

structured_text = llm_response.message.content
print("\n=== Structured JSON Output (LLM Raw) ===")
print(structured_text)

# Step 3: Pydantic validation + fallback repair
table = coerce_table_with_pydantic(structured_text)
normalized_rows = [row_to_schema_dict(r) for r in table.rows]

print("\n=== Final Structured Rows ===")
print(json.dumps({"rows": normalized_rows}, indent=2, ensure_ascii=True))

=== OCR Raw Output ===
Here's the extracted text from the image, exactly as it appears:

| Prod date | Exp date | Product Name | Batch No. | pH 1% (1g/99ml) | Specification |
|---|---|---|---|---|---|
| 04.11.23 | 04.11.24 | DB2 | Stabiliser | 893 | 6.0-8.0 |
| 04.11.23 | 04.11.24 | DB2 | Stabiliser | 894 | 7.17 |
| 04.11.23 | 04.11.24 | DB2 | Stabiliser | 895 | 7.16 |
| 04.11.23 | 04.11.24 | DB2 | Stabiliser | 896 | 7.17 |
| 04.11.23 | 04.11.24 | DB2 | Stabiliser | 897 | 7.13 |
| 04.11.23 | 04.11.24 | DB2 | Stabiliser | 898 | 7.19 |

=== Structured JSON Output (LLM Raw) ===
{
"rows": [
  {
    "Prod date": "04.11.23",
    "Exp date": "04.11.24",
    "Product Name": "DB2",
    "Batch No.": "Stabiliser",
    "pH 1% (1g/99ml)": "7.17",
    "Specification": "6.0-8.0"
  },
  {
    "Prod date": "04.11.23",
    "Exp date": "04.11.24",
    "Product Name": "DB2",
    "Batch No.": "Stabiliser",
    "pH 1% (1g/99ml)": "7.16",
    "Specification": "7.0-8.0"
  },
  {
    "Prod date": "04.11.23",
 

## VLM-only experiment (no OCR stage)
This experiment uses only `qwen3-vl:8b` to read the image and return structured JSON in the same output schema.

In [8]:
import subprocess

VLM_MODEL = "qwen3-vl:8b"
installed_models = subprocess.check_output(["ollama", "ls"], text=True)

if VLM_MODEL in installed_models:
    print(f"Model available: {VLM_MODEL}")
else:
    print(f"Model missing: {VLM_MODEL}")
    print("Run this once in terminal:")
    print(f"  ollama pull {VLM_MODEL}")

Model missing: qwen3-vl:8b
Run this once in terminal:
  ollama pull qwen3-vl:8b


In [7]:
# Direct VLM extraction: no OCR intermediate step
vlm_response = safe_chat(
    model=VLM_MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You extract structured table rows directly from an image. "
                "Return ONLY valid JSON (no markdown, no explanation). "
                "Use this exact schema with exact keys and same case: "
                "{\"rows\": [{\"Prod date\": null, \"Exp date\": null, \"Product Name\": null, \"Batch No.\": null, \"pH\": null}]}. "
                "If multiple rows exist, include all rows. "
                "If a field is missing or unclear, set it to null."
            ),
        },
        {
            "role": "user",
            "content": "Read the table from this image and return rows in the schema.",
            "images": [image_path],
        },
    ],
    options=VISION_OPTIONS,
    retries=3,
 )

vlm_structured_text = vlm_response.message.content
print("=== VLM-only Structured JSON Output (Raw) ===")
print(vlm_structured_text)

vlm_table = coerce_table_with_pydantic(vlm_structured_text)
vlm_normalized_rows = [row_to_schema_dict(r) for r in vlm_table.rows]

print("\n=== VLM-only Final Structured Rows ===")
print(json.dumps({"rows": vlm_normalized_rows}, indent=2, ensure_ascii=True))

ResponseError: model 'qwen3-vl:8b' not found (status code: 404)